In [65]:
import os 
# os.chdir("/hpc/home/ephdh/workspace/suzhou_false_validation/data")

import glob
import pydicom
import numpy as np 
import pandas as pd
import re
from tqdm import tqdm
from datetime import datetime
from tableone import TableOne
from collections import Counter
from pydicom.misc import is_dicom

# Re-organize meta data

In [66]:
tn_excel = pd.read_excel('meta_data/真阴性名单.xlsx', sheet_name='True Negatives')
tn_excel['Group'] = 'TN'
tn_excel = tn_excel.rename(columns={'Patient Name': 'PatientName', 
                                    'Patient Age': 'PatientAge',
                                    'Density category': 'DensityCategory',
                                    'Lesion type': 'LesionType', 
                                    'BI-RADS risk': 'BIRADSRisk', 
                                    'Histological subtype': 'HistologicalSubtype'
                                    }
                                    )

In [67]:
tn_excel.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group
0,徐琴芳,51,C,肿块,3.0,乳腺病,TN
1,沈诗怡,23,C,肿块,3.0,纤维腺瘤伴腺病,TN
2,王琴,42,NaN,NaN,NaN,请剔除,TN
3,龙菲菲,39,C,肿块,3.0,纤维腺瘤,TN
4,张菊,42,C,肿块,3.0,乳腺病伴局部导管上皮增生,TN


In [69]:
tp_excel = pd.read_excel('meta_data/真阳性名单钼靶.xlsx', sheet_name='True Positives')
tp_excel = tp_excel.rename(columns={'PatientName': 'PatientName', 
                                    'PatientAge': 'PatientAge',
                                    'Density category': 'DensityCategory',
                                    ' Lesion type': 'LesionType', 
                                    ' BI-RADS risk': 'BIRADSRisk', 
                                    'Histological subtypes': 'HistologicalSubtype'
                                    }
                                    )
include_cols = ['PatientName', 'PatientAge', 'DensityCategory', 'LesionType', 'BIRADSRisk', 
                'HistologicalSubtype']
tp_excel = tp_excel[include_cols]
tp_excel['Group'] = 'TP'

In [70]:
tp_excel.columns

Index(['PatientName', 'PatientAge', 'DensityCategory', 'LesionType',
       'BIRADSRisk', 'HistologicalSubtype', 'Group'],
      dtype='object')

In [71]:
tp_excel.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group
0,崔敏（左）,41,c,肿块伴簇状钙化,4B,浸润性导管癌,TP
1,崔敏（右）,41,c,肿块,4A,浸润性导管癌,TP
2,顾璟怡,69,c,肿块伴簇状钙化,6,浸润性导管癌,TP
3,李彩琴,58,c,结构紊乱,4C,大汗腺型导管原位癌,TP
4,付启红,55,c,肿块,4B,混合性腺癌,TP


In [72]:
fn_excel = pd.read_excel('meta_data/假阴性名单.xls', sheet_name='original_dcm_info')
fn_excel = fn_excel.rename(columns={'PatientName': 'PatientName', 
                                    'PatientAge': 'PatientAge',
                                    'Density category': 'DensityCategory',
                                    ' Lesion type': 'LesionType', 
                                    ' BI-RADS risk': 'BIRADSRisk', 
                                    'Histological subtypes': 'HistologicalSubtype'
                                    }
                                    )
fn_excel = fn_excel[include_cols]
fn_excel['Group'] = 'FN'

In [ ]:
# print(fn_excel.columns)
fn_excel.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group
0,吴峰,35,d,肿块,2.0,浸润性导管癌,FN
1,严翠芬,49,c,无,3.0,浸润性导管癌,FN
2,吴妹林,68,a,结节,3.0,浸润性导管癌,FN
3,石根娣,64,b,无,1.0,粘液癌,FN
4,贾荣妹,45,b,无,2.0,浸润性导管癌,FN


In [108]:

fp_excel = pd.read_excel('meta_data/假阳性名单.xlsx', sheet_name='Sheet1')
fp_excel.columns = fp_excel.columns.astype(str).str.strip()

fp_excel = fp_excel.rename(columns={
    'Patient Name': 'PatientName',
    'Patient Age': 'PatientAge',
    'Density category': 'DensityCategory',
    'Lesion type': 'LesionType',
    'BI-RADS risk': 'BIRADSRisk',
    'Histological subtype': 'HistologicalSubtype',
    '备注': 'ExclusionRemark',
    'Remark': 'ExclusionRemark',
    'Remarks': 'ExclusionRemark'
})

# 自动寻找备注列
if 'ExclusionRemark' not in fp_excel.columns:
    remark_candidates = [
        col for col in fp_excel.columns
        if fp_excel[col].astype('string').str.contains(r'建议剔除|请剔除', na=False).any()
    ]
    assert len(remark_candidates) == 1, f'Expected exactly one FP remark column, found: {remark_candidates}'
    fp_excel = fp_excel.rename(columns={remark_candidates[0]: 'ExclusionRemark'})

# 根据备注自动识别应剔除病例
fp_raw_count = len(fp_excel)
fp_manual_exclusion_mask = fp_excel['ExclusionRemark'].astype('string').str.contains(
    r'建议剔除|请剔除', na=False
)

fp_exclusion_log = fp_excel.loc[
    fp_manual_exclusion_mask,
    ['PatientName', 'HistologicalSubtype', 'ExclusionRemark']
].copy()

assert fp_raw_count == 203, f'Expected 203 raw FP cases, got {fp_raw_count}'
assert len(fp_exclusion_log) == 3, f'Expected 3 manually excluded FP cases, got {len(fp_exclusion_log)}'

fp_excel = fp_excel.loc[
    ~fp_manual_exclusion_mask,
    include_cols + ['ExclusionRemark']
].copy()

assert len(fp_excel) == 200
fp_excel['Group'] = 'FP'

In [109]:
print(f'FP cases: raw={fp_raw_count}, manually excluded={len(fp_exclusion_log)}, retained={len(fp_excel)}')
display(fp_exclusion_log)
fp_excel.head()

FP cases: raw=203, manually excluded=3, retained=200


,PatientName,HistologicalSubtype,ExclusionRemark
26,贺爱云,导管内乳头状瘤,患者半年后诊断导管原位癌，建议剔除
79,余龙花,导管内乳头状瘤伴低级别导管原位癌,术后免疫病理为导管原位癌，请剔除
101,周静,浸润性导管癌,术后免疫病理为浸润性导管癌，请剔除


,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,ExclusionRemark,Group
0,崔梅巧,42,C,肿块影,4A,导管内乳头状瘤伴大汗腺化生,NaN,FP
1,陈蔚娟,50,C,成簇钙化灶,4A,乳腺病伴导管扩张，部分导管上皮不典型增生伴钙盐沉积，柱状细胞变性,NaN,FP
2,程芬,35,C,肿块影、非对称影，伴细点状模糊钙化,4A,纤维腺瘤,NaN,FP
3,梁素兰,55,C,结节样致密影,4A,乳腺病伴囊肿形成,NaN,FP
4,郑春芳,54,B,肿块,4A,导管内乳头状瘤伴大汗腺化生,NaN,FP


In [110]:
meta_data = pd.concat([tp_excel, tn_excel, fp_excel, fn_excel], ignore_index=True)
meta_data = meta_data[meta_data['HistologicalSubtype']!='请剔除'].reset_index(drop=True)

In [111]:
meta_data.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,ExclusionRemark
0,崔敏（左）,41,c,肿块伴簇状钙化,4B,浸润性导管癌,TP,NaN
1,崔敏（右）,41,c,肿块,4A,浸润性导管癌,TP,NaN
2,顾璟怡,69,c,肿块伴簇状钙化,6,浸润性导管癌,TP,NaN
3,李彩琴,58,c,结构紊乱,4C,大汗腺型导管原位癌,TP,NaN
4,付启红,55,c,肿块,4B,混合性腺癌,TP,NaN


In [112]:
print(meta_data['Group'].value_counts())

Group
TN    265
TP    252
FN    209
FP    200
Name: count, dtype: int64


In [113]:
# print(meta_data['HistologicalSubtype'].value_counts())

In [114]:
print(meta_data.shape, meta_data['PatientName'].nunique())

(926, 8) 917


In [116]:
duplidcated_meta = meta_data[meta_data.duplicated(subset=['PatientName'], keep=False)].sort_values(
    by=['PatientName']
).reset_index(drop=True)

# 打印重复记录总体情况
print(
    "重复记录表 shape:",
    duplidcated_meta.shape,
    "重复姓名数量:",
    duplidcated_meta['PatientName'].nunique()
)

# 按 PatientName 分组，把同名患者的完整信息放在一起显示
with pd.option_context('display.max_columns', None):
    for name, group in duplidcated_meta.groupby('PatientName'):
        print(f"\n===== PatientName: {name} =====")
        display(group)

重复记录表 shape: (18, 8) 重复姓名数量: 9

===== PatientName: 丁红 =====


,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,ExclusionRemark
0,丁红,46,d,无,2.0,浸润性导管癌,FN,NaN
1,丁红,52,b,肿块,4A,浸润性导管癌,TP,NaN



===== PatientName: 何冬琴 =====


,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,ExclusionRemark
2,何冬琴,41,D,肿块,4A,纤维腺瘤,FP,NaN
3,何冬琴,42,C,肿块,3.0,纤维腺瘤,TN,NaN



===== PatientName: 刘桂华 =====


...

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,ExclusionRemark
12,张美华,42,c,无,2.0,浸润性导管癌,FN,NaN
13,张美华,61,c,肿块,4C,非特殊型浸润癌,TP,NaN



===== PatientName: 武利敏 =====


,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,ExclusionRemark
14,武利敏,43,C,肿块,3.0,纤维腺瘤,TN,NaN
15,武利敏,42,C,肿块,4B,纤维腺瘤,FP,NaN



===== PatientName: 金丽平 =====


,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,ExclusionRemark
16,金丽平,43,C,肿块伴钙化,4A,乳腺病伴纤维腺瘤形成趋势,FP,NaN
17,金丽平,41,c,肿块,4A,导管原位癌,TP,NaN


In [ ]:
# print(non_duplidcated_meta['Group'].value_counts())

In [ ]:
# duplidcated_meta

In [117]:
# 保存原始 metadata + 定义路径
from pathlib import Path
import shutil
import pandas as pd

META_DIR = Path("meta_data")
META_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = META_DIR / "meta_data_raw.csv"
EDITED_PATH = META_DIR / "meta_data_raw_edited.csv"
EDIT_LOG_PATH = META_DIR / "meta_data_raw_edit_log.csv"

# 旧版本只备份一次
BACKUP_PATH = META_DIR / "meta_data_raw_edited_before_rebuild_2026-09-23.csv"

meta_data.to_csv(RAW_PATH, index=False, encoding="utf-16")

meta_data_edited = pd.read_csv(RAW_PATH, encoding="utf-16")

print("Raw metadata shape:", meta_data_edited.shape)
print("\nGroup counts:")
print(meta_data_edited["Group"].value_counts())

Raw metadata shape: (926, 8)

Group counts:
Group
TN    265
TP    252
FN    209
FP    200
Name: count, dtype: int64


In [118]:
required_columns = {
    "PatientName",
    "PatientAge",
    "Group",
}

missing_columns = required_columns - set(meta_data_edited.columns)
assert not missing_columns, (f"Missing required columns: {missing_columns}")

# Group
VALID_GROUPS = {"TP", "TN", "FP", "FN"}

unexpected_groups = (
    set(meta_data_edited["Group"].dropna().unique())
    - VALID_GROUPS
)

assert not unexpected_groups, (
    f"Unexpected groups found: {unexpected_groups}"
)

EXCLUSION_PATTERN = r"建议剔除|请剔除"
if "ExclusionRemark" in meta_data_edited.columns:
    exclusion_mask = (meta_data_edited["ExclusionRemark"].fillna("").astype(str)
        .str.contains(
            EXCLUSION_PATTERN,
            regex=True
        )
    )
    if exclusion_mask.any():
        remaining_excluded = meta_data_edited.loc[
            exclusion_mask,
            [
                "PatientName",
                "PatientAge",
                "Group",
                "ExclusionRemark",
            ]
        ]
        display(remaining_excluded)
        raise AssertionError(
            "Records marked for exclusion still remain "
            "in meta_data_raw.csv."
        )

print("Basic metadata checks passed.")

Basic metadata checks passed.


In [119]:
meta_data_edited["OriginalPatientName"] = (
    meta_data_edited["PatientName"]
)
edit_log = []
n_before_manual_edit = len(meta_data_edited)

MANUAL_REMOVE = [
    {
        "Group": "TP",
        "PatientName": "崔敏（右）",
        "PatientAge": 41,
        "Reason": "人工确认左右侧记录需要合并，删除右侧记录",
    },
]

for rule in MANUAL_REMOVE:
    age = pd.to_numeric(
        meta_data_edited["PatientAge"],
        errors="coerce"
    )
    mask = (
        (meta_data_edited["Group"] == rule["Group"])
        & (
            meta_data_edited["PatientName"]
            == rule["PatientName"]
        )
        & (age == rule["PatientAge"])
    )
    match_count = mask.sum()
    assert match_count == 1, (
        f"Manual REMOVE expected 1 match, "
        f"but found {match_count}: {rule}"
    )
    row = meta_data_edited.loc[mask].iloc[0]
    edit_log.append({
        "Action": "REMOVE",
        "Group": row["Group"],
        "OriginalPatientName": row["PatientName"],
        "EditedPatientName": "",
        "PatientAge": row["PatientAge"],
        "Reason": rule["Reason"],
    })
    meta_data_edited = (
        meta_data_edited.loc[~mask]
        .reset_index(drop=True)
    )


MANUAL_RENAME = [
    {
        "Group": "TP",
        "PatientName": "崔敏（左）",
        "PatientAge": 41,
        "NewPatientName": "崔敏",
        "Reason": "删除右侧记录后，左侧记录恢复患者姓名",
    },
]


for rule in MANUAL_RENAME:

    age = pd.to_numeric(
        meta_data_edited["PatientAge"],
        errors="coerce"
    )

    mask = (
        (meta_data_edited["Group"] == rule["Group"])
        & (
            meta_data_edited["PatientName"]
            == rule["PatientName"]
        )
        & (age == rule["PatientAge"])
    )

    match_count = mask.sum()

    assert match_count == 1, (
        f"Manual RENAME expected 1 match, "
        f"but found {match_count}: {rule}"
    )

    edit_log.append({
        "Action": "RENAME",
        "Group": rule["Group"],
        "OriginalPatientName": rule["PatientName"],
        "EditedPatientName": rule["NewPatientName"],
        "PatientAge": rule["PatientAge"],
        "Reason": rule["Reason"],
    })

    meta_data_edited.loc[
        mask,
        "PatientName"
    ] = rule["NewPatientName"]


print(
    "Manual edits completed:",
    n_before_manual_edit,
    "->",
    len(meta_data_edited)
)

display(pd.DataFrame(edit_log))

Manual edits completed: 926 -> 925


,Action,Group,OriginalPatientName,EditedPatientName,PatientAge,Reason
0,REMOVE,TP,崔敏（右）,,41,人工确认左右侧记录需要合并，删除右侧记录
1,RENAME,TP,崔敏（左）,崔敏,41,删除右侧记录后，左侧记录恢复患者姓名


In [120]:
# 处理所有重复命名
duplicate_mask = (meta_data_edited["PatientName"].duplicated(keep=False))

duplicate_rows = (meta_data_edited.loc[duplicate_mask].copy())

print("重复记录数量:", len(duplicate_rows))
print("重复姓名组数:",duplicate_rows["PatientName"].nunique())

if not duplicate_rows.empty:
    display(
        duplicate_rows[
            [
                "PatientName",
                "PatientAge",
                "Group",
            ]
        ].sort_values(
            ["PatientName", "Group"]
        )
    )

same_name_group_count = (meta_data_edited.groupby(["PatientName", "Group"])["PatientName"].transform("size"))
same_name_group_order = (meta_data_edited.groupby(["PatientName", "Group"],sort=False).cumcount()+ 1)

for idx in meta_data_edited.index[duplicate_mask]:
    old_name = meta_data_edited.at[idx,"PatientName"]
    group = meta_data_edited.at[idx,"Group"]
    age = meta_data_edited.at[idx,"PatientAge"]
    # 1
    if same_name_group_count.loc[idx] == 1:
        new_name = f"{old_name}_{group}"
    # 2
    else:
        order = same_name_group_order.loc[idx]
        new_name = (f"{old_name}_{group}{order}")

    edit_log.append({
        "Action": "AUTO_RENAME_DUPLICATE",
        "Group": group,
        "OriginalPatientName": old_name,
        "EditedPatientName": new_name,
        "PatientAge": age,
        "Reason": "Automatically resolve duplicated PatientName",
    })

    meta_data_edited.at[idx,"PatientName"] = new_name

remaining_duplicates = meta_data_edited[meta_data_edited["PatientName"].duplicated(keep=False)]
assert remaining_duplicates.empty, ("Duplicated PatientName still exists:\n"f"{remaining_duplicates}")
print("\nDuplicate-name processing completed.")

auto_rename_log = pd.DataFrame([
    row for row in edit_log
    if row["Action"] == "AUTO_RENAME_DUPLICATE"
])

display(auto_rename_log)

重复记录数量: 18
重复姓名组数: 9


,PatientName,PatientAge,Group
813,丁红,46,FN
224,丁红,52,TP
561,何冬琴,41,FP
508,何冬琴,42,TN
542,刘桂华,59,FP
117,刘桂华,45,TP
78,刘颖,47,TP
228,刘颖,26,TP
768,周素英,52,FN
89,周素英,59,TP



Duplicate-name processing completed.


,Action,Group,OriginalPatientName,EditedPatientName,PatientAge,Reason
0,AUTO_RENAME_DUPLICATE,TP,张美华,张美华_TP,61,Automatically resolve duplicated PatientName
1,AUTO_RENAME_DUPLICATE,TP,刘颖,刘颖_TP1,47,Automatically resolve duplicated PatientName
2,AUTO_RENAME_DUPLICATE,TP,周素英,周素英_TP,59,Automatically resolve duplicated PatientName
3,AUTO_RENAME_DUPLICATE,TP,刘桂华,刘桂华_TP,45,Automatically resolve duplicated PatientName
4,AUTO_RENAME_DUPLICATE,TP,金丽平,金丽平_TP,41,Automatically resolve duplicated PatientName
5,AUTO_RENAME_DUPLICATE,TP,丁红,丁红_TP,52,Automatically resolve duplicated PatientName
6,AUTO_RENAME_DUPLICATE,TP,刘颖,刘颖_TP2,26,Automatically resolve duplicated PatientName
7,AUTO_RENAME_DUPLICATE,TN,武利敏,武利敏_TN,43,Automatically resolve duplicated PatientName
8,AUTO_RENAME_DUPLICATE,TN,季雅情,季雅情_TN,39,Automatically resolve duplicated PatientName
9,AUTO_RENAME_DUPLICATE,TN,何冬琴,何冬琴_TN,42,Automatically resolve duplicated PatientName


In [121]:
assert meta_data_edited["PatientName"].is_unique, (
    "PatientName is not unique."
)

assert meta_data_edited["PatientName"].notna().all(), (
    "Missing PatientName found."
)

assert meta_data_edited["Group"].notna().all(), (
    "Missing Group found."
)

expected_final_size = (
    n_before_manual_edit
    - len(MANUAL_REMOVE)
)

assert len(meta_data_edited) == expected_final_size, (
    f"Unexpected final size: {len(meta_data_edited)}, "
    f"expected {expected_final_size}"
)

if "ExclusionRemark" in meta_data_edited.columns:
    meta_data_edited = meta_data_edited.drop(
        columns=["ExclusionRemark"]
    )

print("Final metadata shape:", meta_data_edited.shape)

print("\nFinal group counts:")
print(meta_data_edited["Group"].value_counts())

meta_data_edited.to_csv(
    EDITED_PATH,
    index=False,
    encoding="utf-16",
    sep="\t"
)

edit_log_df = pd.DataFrame(edit_log)

edit_log_df.to_csv(
    EDIT_LOG_PATH,
    index=False,
    encoding="utf-16",
    sep="\t"
)

print("\n保存完成：")
print(EDITED_PATH)

print("\n审计日志：")
print(EDIT_LOG_PATH)

Final metadata shape: (925, 8)

Final group counts:
Group
TN    265
TP    251
FN    209
FP    200
Name: count, dtype: int64

保存完成：
meta_data/meta_data_raw_edited.csv

审计日志：
meta_data/meta_data_raw_edit_log.csv


## Standardize and categorize the columns

In [122]:
meta_data_edited.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,OriginalPatientName
0,崔敏,41,c,肿块伴簇状钙化,4B,浸润性导管癌,TP,崔敏（左）
1,顾璟怡,69,c,肿块伴簇状钙化,6,浸润性导管癌,TP,顾璟怡
2,李彩琴,58,c,结构紊乱,4C,大汗腺型导管原位癌,TP,李彩琴
3,付启红,55,c,肿块,4B,混合性腺癌,TP,付启红
4,周夏琴,49,b,细小钙化灶,4C,导管原位癌伴微小浸润,TP,周夏琴


In [123]:
# Standardize Density

# Mapping to a canonical A/B/C/D code
density_map = {
    "a": "A", "A": "A",
    "b": "B", "B": "B",
    "c": "C", "C": "C",
    "d": "D", "D": "D",
    "中量腺体型": "B",   # adjust if you prefer C
    "混合型": "B",       # could also be C depending on your radiologist
    "a/b": "B",         # boundary -> choose B as intermediate
    "无": np.nan
}

def standardize_density(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return density_map.get(x, np.nan)

meta_data_edited["DensityCategory_std"] = meta_data_edited["DensityCategory"].apply(standardize_density)

# Optional: also map A/B/C/D -> 1/2/3/4 for modeling
density_numeric_map = {"A": 1, "B": 2, "C": 3, "D": 4}
meta_data_edited["DensityCategory_num"] = meta_data_edited["DensityCategory_std"].map(density_numeric_map)

# print(meta_data_edited[["DensityCategory", "DensityCategory_std", "DensityCategory_num"]].head(20))
print("DensityCategory_std value counts:")
print(meta_data_edited["DensityCategory_std"].value_counts(dropna=False))

DensityCategory_std value counts:
DensityCategory_std
C      654
B      137
D      101
A       31
NaN      2
Name: count, dtype: int64


In [124]:
# Standardize BIRADS
def standardize_birads(x):
    """
    Return main BIRADS category as an integer (0-6), or np.nan.
    """
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()

    # Handle 4A/4B/4C
    if x in ["4A", "4B", "4C"]:
        return 4

    # Excel/CSV may represent an integer BI-RADS value as "3.0".
    try:
        v = float(x)
    except ValueError:
        return np.nan
    return int(v) if v.is_integer() and 0 <= v <= 6 else np.nan

def birads_subtype(x):
    """
    Extract subtype for BIRADS 4 (e.g., 4A/4B/4C), otherwise return None.
    """
    if pd.isna(x):
        return None
    x = str(x).strip().upper()
    if x in ["4A", "4B", "4C"]:
        return x
    return None

meta_data_edited["BIRADSRisk_std"] = meta_data_edited["BIRADSRisk"].apply(standardize_birads).astype("float")
meta_data_edited["BIRADSRisk_sub"] = meta_data_edited["BIRADSRisk"].apply(birads_subtype)

print("BIRADSRisk_std value counts:")
print(meta_data_edited["BIRADSRisk_std"].value_counts(dropna=False).sort_index())

BIRADSRisk_std value counts:
BIRADSRisk_std
0.0      2
1.0     37
2.0    134
3.0    301
4.0    421
5.0     23
6.0      5
NaN      2
Name: count, dtype: int64


In [125]:
# Categorize LesionType

def categorize_lesion_type(x: str) -> str:
    if pd.isna(x):
        return "Unknown/NA"
    x = str(x).strip()

    # Explicit normal/benign
    if x in ["无异常", "无"] or "良性" in x or "增生" in x:
        return "Negative/Benign"

    # Simple placeholder or missing
    if x in ["-", "—"]:
        return "Unknown/NA"

    # Keywords (order matters: more specific patterns first)
    has_mass   = ("肿块" in x) or ("肿块影" in x)
    has_nodule = ("结节" in x) or ("结节影" in x) or ("结节样" in x)
    has_calc   = ("钙化" in x) or ("钙化灶" in x)
    has_arch   = ("结构紊乱" in x) or ("结构扭曲" in x) or ("结构稍乱" in x)
    has_asym   = ("不对称" in x) or ("非对称" in x) or ("局灶性不对称" in x) or ("局灶不对称" in x)
    has_dense  = ("致密影" in x) or ("高密度影" in x)

    # Mass / Nodule dominant
    if has_mass and not (has_calc or has_arch or has_asym or has_dense):
        return "Mass"
    if has_nodule and not (has_calc or has_arch or has_asym or has_dense):
        return "Nodule/FocalDensity"

    # Calcification dominant (no strong mass/nodule/arch/asym)
    if has_calc and not (has_mass or has_nodule or has_arch or has_asym):
        return "Calcification"

    # Architectural distortion / asymmetry dominant
    if (has_arch or has_asym or has_dense) and not (has_mass or has_nodule or has_calc):
        return "ArchitecturalDistortion/Asymmetry"

    # Mixed lesions
    if sum([has_mass, has_nodule, has_calc, has_arch, has_asym or has_dense]) >= 2:
        return "Mixed"

    # Fallbacks
    if has_mass:
        return "Mass"
    if has_nodule:
        return "Nodule/FocalDensity"
    if has_calc:
        return "Calcification"
    if has_arch or has_asym or has_dense:
        return "ArchitecturalDistortion/Asymmetry"

    return "Other"

# 2. Apply categorization
meta_data_edited["LesionType_cat"] = meta_data_edited["LesionType"].apply(categorize_lesion_type)

# 3. Inspect result
print("Category counts:")
print(meta_data_edited["LesionType_cat"].value_counts())

Category counts:
LesionType_cat
Mass                                 370
Calcification                        185
Mixed                                165
Negative/Benign                       79
Nodule/FocalDensity                   53
ArchitecturalDistortion/Asymmetry     43
Unknown/NA                            29
Other                                  1
Name: count, dtype: int64


In [126]:
# Recorded histology in the TP/FN cohort: four mutually exclusive categories.
# This descriptive variable never changes Group, GroundTruth, or model scores.
import re
import unicodedata

HISTOLOGY_CATEGORIES = [
    "IDC / NST", "DCIS", "Other malignancies",
    "Histology unavailable / uncertain",
]
HISTOLOGY_NOT_APPLICABLE = "Not applicable (TN/FP)"

# Remove columns left in memory by runs of the retired detailed classifier.
# This also makes rerunning this cell produce only the current four-class output.
LEGACY_HISTOLOGY_COLUMNS = [
    "Histology_normalized",
    "Histology_primary_cat",
    "Histology_table1_cat",
    "Histology_component_labels",
    "Histology_is_multicomponent",
    "Histology_review_flag",
    "Histology_review_reason",
    "Histology_coarse",
]
meta_data_edited.drop(columns=LEGACY_HISTOLOGY_COLUMNS, errors="ignore", inplace=True)


def normalize_histology(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).strip()
    text = re.sub(r"\s+", "", text).rstrip("。.;；")
    return text.replace("粘液", "黏液")


# IDC with accompanying DCIS remains IDC; DCIS with microinvasion is Other.
# Named, independent invasive types (e.g. IDC + ILC) are Other. New wording
# remains uncertain until it is explicitly mapped, never guessed by keyword.
HISTOLOGY_TEXT_TO_CATEGORY = {
    # IDC / NST
    '以导管原位癌为主的浸润性导管癌': 'IDC / NST',
    '导管原位癌为主的浸润性导管癌': 'IDC / NST',
    '导管原位癌伴浸润性导管癌(6mm)': 'IDC / NST',
    '浸润性导管癌': 'IDC / NST',
    '浸润性导管癌伴黏液分泌': 'IDC / NST',
    '非特殊型浸润性癌': 'IDC / NST',
    '非特殊型浸润癌': 'IDC / NST',
    # DCIS
    '低级别大汗腺型导管原位癌': 'DCIS',
    '低级别导管原位癌': 'DCIS',
    '大汗腺型导管原位癌': 'DCIS',
    '导管内癌': 'DCIS',
    '导管原位癌': 'DCIS',
    '高级别导管原位癌': 'DCIS',
    # Other malignancies
    '乳头Paget病': 'Other malignancies',
    '乳头Paget病伴导管原位癌': 'Other malignancies',
    '伴大汗腺化生的浸润性癌': 'Other malignancies',
    '伴导管原位癌的黏液腺癌': 'Other malignancies',
    '伴间叶分化的化生性癌': 'Other malignancies',
    '包裹性乳头状癌': 'Other malignancies',
    '包裹性乳头状癌混合浸润性导管癌': 'Other malignancies',
    '化生性癌,化生成分为鳞状细胞癌': 'Other malignancies',
    '原位型乳头状癌': 'Other malignancies',
    '多形性浸润性小叶癌': 'Other malignancies',
    '实性乳头状癌': 'Other malignancies',
    '实性乳头状癌伴多灶微小浸润': 'Other malignancies',
    '实性乳头状癌伴间质浸润': 'Other malignancies',
    '导管内乳头状癌伴浸润性导管癌': 'Other malignancies',
    '导管内实性乳头状癌': 'Other malignancies',
    '导管原位癌伴微小浸润': 'Other malignancies',
    '导管原位癌伴微小间质浸润': 'Other malignancies',
    '导管原位癌伴微浸润': 'Other malignancies',
    '导管原位癌伴间质微小浸润': 'Other malignancies',
    '导管原位癌伴间质浸润': 'Other malignancies',
    '浸润性乳头状癌': 'Other malignancies',
    '浸润性大汗腺癌': 'Other malignancies',
    '浸润性实性乳头状癌': 'Other malignancies',
    '浸润性实性乳头状癌合并黏液癌': 'Other malignancies',
    '浸润性小叶癌': 'Other malignancies',
    '浸润性小叶癌(左)': 'Other malignancies',
    '浸润性小管癌': 'Other malignancies',
    '浸润性微乳头状癌': 'Other malignancies',
    '浸润性癌': 'Other malignancies',
    '浸润性癌伴大汗腺化生': 'Other malignancies',
    '浸润性癌伴神经内分泌分化': 'Other malignancies',
    '混合型黏液癌,部分浸润性导管癌': 'Other malignancies',
    '混合性浸润性癌': 'Other malignancies',
    '混合性腺癌': 'Other malignancies',
    '混合浸润性导管癌及浸润性微乳头状癌': 'Other malignancies',
    '癌肉瘤(腺癌15%+平滑肌肉瘤85%)': 'Other malignancies',
    '部分浸润性导管癌、部分浸润性小叶癌': 'Other malignancies',
    '黏液癌': 'Other malignancies',
    '黏液腺癌': 'Other malignancies',
    # Histology unavailable / uncertain
    '': 'Histology unavailable / uncertain',
    '低分化腺癌组织,考虑乳腺来源': 'Histology unavailable / uncertain',
    '导管内乳头状病变,实性乳头状癌不能除外': 'Histology unavailable / uncertain',
    '导管内乳头状瘤伴导管上皮增生活跃': 'Histology unavailable / uncertain',
    '我院未手术': 'Histology unavailable / uncertain',
    '无手术病理': 'Histology unavailable / uncertain',
    '纤维腺瘤及乳腺病': 'Histology unavailable / uncertain',
}

HISTOLOGY_UNCERTAIN_REASONS = {
    "": "Histology text missing",
    "无手术病理": "No histological diagnosis recorded in this field",
    "我院未手术": "No histological diagnosis recorded in this field",
    "纤维腺瘤及乳腺病": "Benign wording conflicts with TP/FN cohort label",
    "导管内乳头状瘤伴导管上皮增生活跃":
        "Benign wording conflicts with TP/FN cohort label",
    "导管内乳头状病变,实性乳头状癌不能除外":
        "Malignancy is not confirmed in this field",
    "低分化腺癌组织,考虑乳腺来源":
        "Carcinoma is recorded, but breast origin is uncertain",
}

histology_lookup = {}
for wording, category in HISTOLOGY_TEXT_TO_CATEGORY.items():
    key = normalize_histology(wording)
    assert key not in histology_lookup, f"Duplicate histology mapping: {key}"
    histology_lookup[key] = category


def classify_histology(value, group):
    if group not in {"TP", "FN"}:
        return HISTOLOGY_NOT_APPLICABLE, ""
    key = normalize_histology(value)
    category = histology_lookup.get(key, "Histology unavailable / uncertain")
    if category != "Histology unavailable / uncertain":
        return category, ""
    reason = HISTOLOGY_UNCERTAIN_REASONS.get(
        key, "Unmapped or unsupported histology wording"
    )
    return category, reason


histology_fields = [
    classify_histology(value, group)
    for value, group in zip(
        meta_data_edited["HistologicalSubtype"], meta_data_edited["Group"]
    )
]
meta_data_edited[["Histology_category", "Histology_uncertain_reason"]] = (
    pd.DataFrame(
        histology_fields,
        columns=["Histology_category", "Histology_uncertain_reason"],
        index=meta_data_edited.index,
    )
)
print("Recorded histology among TP/FN cases:")
print(
    meta_data_edited.loc[
        meta_data_edited["Group"].isin(["TP", "FN"]), "Histology_category"
    ].value_counts(dropna=False).reindex(HISTOLOGY_CATEGORIES, fill_value=0)
)


Recorded histology among TP/FN cases:
Histology_category
IDC / NST                            343
DCIS                                  24
Other malignancies                    86
Histology unavailable / uncertain      7
Name: count, dtype: int64


In [127]:
meta_data_edited.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,OriginalPatientName,DensityCategory_std,DensityCategory_num,BIRADSRisk_std,BIRADSRisk_sub,LesionType_cat,Histology_category,Histology_uncertain_reason
0,崔敏,41,c,肿块伴簇状钙化,4B,浸润性导管癌,TP,崔敏（左）,C,3.0,4.0,4B,Mixed,IDC / NST,
1,顾璟怡,69,c,肿块伴簇状钙化,6,浸润性导管癌,TP,顾璟怡,C,3.0,6.0,NaN,Mixed,IDC / NST,
2,李彩琴,58,c,结构紊乱,4C,大汗腺型导管原位癌,TP,李彩琴,C,3.0,4.0,4C,ArchitecturalDistortion/Asymmetry,DCIS,
3,付启红,55,c,肿块,4B,混合性腺癌,TP,付启红,C,3.0,4.0,4B,Mass,Other malignancies,
4,周夏琴,49,b,细小钙化灶,4C,导管原位癌伴微小浸润,TP,周夏琴,B,2.0,4.0,4C,Calcification,Other malignancies,


### Coarser categorization

In [128]:
df = meta_data_edited.copy()

# --- Density (coarse) ---

density_map = {
    "a": "A", "A": "A",
    "b": "B", "B": "B",
    "c": "C", "C": "C",
    "d": "D", "D": "D",
    "中量腺体型": "B",   # adjust to C if your radiologists prefer
    "混合型": "B",
    "a/b": "B",
    "无": np.nan
}

def standardize_density(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return density_map.get(x, np.nan)

df["Density_std"] = df["DensityCategory"].apply(standardize_density)

def coarse_density(x):
    if x in ["A", "B"]:
        return "Low density (A/B)"
    if x in ["C", "D"]:
        return "High density (C/D)"
    return "Unknown"

df["Density_coarse"] = df["Density_std"].apply(coarse_density)

In [129]:
# --- BIRADS (coarse) ---

def birads_main(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s in ["4A", "4B", "4C"]:
        return 4
    return standardize_birads(s)

def birads_coarse(x):
    if pd.isna(x):
        return "Unknown"

    main = standardize_birads(x)
    if pd.isna(main):
        return "Unknown"
    mapping = {
        0: "0 Incomplete",
        1: "1–2 Negative/Benign",
        2: "1–2 Negative/Benign",
        3: "3 Probably benign",
        4: "4 Suspicious",
        5: "5 Highly suggestive",
        6: "6 Known malignancy",
    }

    return mapping.get(int(main), "Unknown")

df["BIRADS_main"] = df["BIRADSRisk"].apply(birads_main).astype("float")
df["BIRADS_coarse"] = df["BIRADSRisk"].apply(birads_coarse)

# Optional: binary variable for modeling (>=4 as "positive")
df["BIRADS_binary"] = np.where(df["BIRADS_main"] >= 4, "Positive (≥4)", "Negative (<4)")
df.loc[df["BIRADS_main"].isna(), "BIRADS_binary"] = "Unknown"

In [130]:
# --- LesionType (coarse) ---

def lesiontype_coarse(x: str) -> str:
    if pd.isna(x):
        return "Other/Unknown"
    x = str(x).strip()

    # Explicit negative/benign wording
    if x in ["无异常", "无"] or "良性" in x or "增生" in x:
        return "Negative/Benign/None"

    if x in ["-", "—"]:
        return "Other/Unknown"

    has_mass   = ("肿块" in x) or ("肿块影" in x)
    has_nodule = ("结节" in x) or ("结节影" in x) or ("结节样" in x)
    has_calc   = ("钙化" in x) or ("钙化灶" in x)
    has_arch   = ("结构紊乱" in x) or ("结构扭曲" in x) or ("结构稍乱" in x)
    has_asym   = ("不对称" in x) or ("非对称" in x) or ("局灶性不对称" in x) or ("局灶不对称" in x)
    has_dense  = ("致密影" in x) or ("高密度影" in x)

    # Mixed if ≥2 major components
    major_flags = [has_mass or has_nodule, has_calc, has_arch or has_asym or has_dense]
    if sum(major_flags) >= 2:
        return "Mixed lesions"

    # Single-dominant patterns
    if has_mass or has_nodule:
        return "Mass/Nodule dominant"
    if has_calc:
        return "Calcification dominant"
    if has_arch or has_asym or has_dense:
        return "Architectural distortion/asymmetry"

    return "Other/Unknown"

df["LesionType_coarse"] = df["LesionType"].apply(lesiontype_coarse)

In [131]:
print("Density_coarse:")
print(df["Density_coarse"].value_counts(dropna=False), "\n")

print("BIRADS_coarse:")
print(df["BIRADS_coarse"].value_counts(dropna=False), "\n")

print("LesionType_coarse:")
print(df["LesionType_coarse"].value_counts(dropna=False), "\n")


Density_coarse:
Density_coarse
High density (C/D)    755
Low density (A/B)     168
Unknown                 2
Name: count, dtype: int64 

BIRADS_coarse:
BIRADS_coarse
4 Suspicious           421
3 Probably benign      301
1–2 Negative/Benign    171
5 Highly suggestive     23
6 Known malignancy       5
0 Incomplete             2
Unknown                  2
Name: count, dtype: int64 

LesionType_coarse:
LesionType_coarse
Mass/Nodule dominant                  423
Calcification dominant                185
Mixed lesions                         165
Negative/Benign/None                   79
Architectural distortion/asymmetry     43
Other/Unknown                          30
Name: count, dtype: int64 



In [132]:
meta_data_coarse = df.copy()

In [133]:
# Let's try exclusion

meta_data_excluded = meta_data_coarse[
    meta_data_coarse['DensityCategory_std'].notna() &
    meta_data_coarse['BIRADSRisk_std'].notna()
]

print(meta_data_excluded['Group'].value_counts())

Group
TN    265
TP    251
FN    205
FP    200
Name: count, dtype: int64


In [ ]:
# # Check the file paths 
# def find_all_dicoms(root_dir: str):
#     """
#     Recursively search for all DICOM files using pydicom.misc.is_dicom.
#     Returns a list of verified DICOM file paths.
#     """
#     dicom_files = []
#     for file in glob.glob(os.path.join(root_dir, "**"), recursive=True):
#         if os.path.isfile(file) and is_dicom(file):
#             dicom_files.append(file)
#     return dicom_files

# root_directory = "/data2/dh/MammographyData/SzOriginalFinal_2"
# dicom_files = find_all_dicoms(root_directory)

# print(f"✅ Found {len(dicom_files)} valid DICOM files.")
# # for f in dicom_files[:5]:
# #     print(f)

# dicom_path_dict = {"TP": [], "TN": [], "FP": [], "FN": []}
# for i in dicom_files:
#     dicom_path_dict[i.split("/")[6]].append(i)

In [134]:
# 读取当前目录中的 PNG 图像，并按 TP/TN/FP/FN 分组
root_directory = next(
    (os.path.abspath(path) for path in ('preprocessed_data', 'data/preprocessed_data')
     if os.path.isdir(path)),
    None,
)
if root_directory is None:
    raise FileNotFoundError('未找到 preprocessed_data 目录；请确认 Notebook 的工作目录。')

png_path_dict = {"TP": [], "TN": [], "FP": [], "FN": []}
for folder, _, filenames in os.walk(root_directory):
    group = os.path.relpath(folder, root_directory).split(os.sep)[0]
    if group not in png_path_dict:
        continue
    for filename in filenames:
        if filename.lower().endswith('.png'):
            png_path_dict[group].append(os.path.join(folder, filename))

for paths in png_path_dict.values():
    paths.sort()
png_files = [path for paths in png_path_dict.values() for path in paths]
print(f"✅ Found {len(png_files)} PNG files in {root_directory}.")

# 兼容后续仍使用 dicom_path_dict 的现有单元格
# 该变量现在存放 PNG 路径，而非 DICOM 路径
dicom_path_dict = png_path_dict

✅ Found 3849 PNG files in /Users/hannn/Documents/论文数据/data/preprocessed_data.


In [135]:
print(len(dicom_path_dict['TP']))
print(len(dicom_path_dict['TN']))
print(len(dicom_path_dict['FP']))
print(len(dicom_path_dict['FN']))

990
1001
1070
788


In [136]:
dicom_path_dict['FP']

['/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/丁小燕/11/1_0.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/丁小燕/2/1_0.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/丁小燕/5/1_0.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/丁小燕/8/1_0.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/万家秀/2/1_0.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/万家秀/2/2_1.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/万家秀/2/3_2.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/万家秀/2/6_3.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/严启华/2/1_0.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/严启华/2/2_1.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/严启华/2/3_2.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/严启华/2/4_3.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/于换换（4b）/2/2_0.png',
 '/Users/hannn/Documents/论文数据/data/preprocessed_data/FP/于换换

In [137]:
# Exclude 张美华_FN
for idx, row in meta_data_excluded.iterrows():
    group = row['Group']
    patient_name = row['PatientName']
    dicom_list = dicom_path_dict.get(group, [])
    matched_files = [f for f in dicom_list if patient_name.split("_")[0].split()[0].split("（")[0] in f]
    if not matched_files:
        print(f"⚠️ No DICOM files found for Patient: {patient_name} in Group: {group}")

⚠️ No DICOM files found for Patient: 任伟娟 in Group: TP
⚠️ No DICOM files found for Patient: 朱仙娣 in Group: TN
⚠️ No DICOM files found for Patient: 蒋莹莹 in Group: TN
⚠️ No DICOM files found for Patient: 张雪艳 in Group: TN
⚠️ No DICOM files found for Patient: 余晓燕 in Group: TN
⚠️ No DICOM files found for Patient: 闵雪 in Group: TN
⚠️ No DICOM files found for Patient: 吴珍 in Group: TN
⚠️ No DICOM files found for Patient: 鲁礼媛 in Group: TN
⚠️ No DICOM files found for Patient: 吴珺 in Group: TN
⚠️ No DICOM files found for Patient: 丁可 in Group: TN
⚠️ No DICOM files found for Patient: 李洁 in Group: TN
⚠️ No DICOM files found for Patient: 郐世伟 in Group: TN
⚠️ No DICOM files found for Patient: 范文珺 in Group: TN
⚠️ No DICOM files found for Patient: 耿文静 in Group: TN
⚠️ No DICOM files found for Patient: 张小露 in Group: TN
⚠️ No DICOM files found for Patient: 谈吉敏 in Group: TN
⚠️ No DICOM files found for Patient: 潘小芹 in Group: FN
⚠️ No DICOM files found for Patient: 刘志敏 in Group: FN
⚠️ No DICOM files found for Patie

In [138]:
# Exclude cases without a PNG image in the matching group/patient folder.
import re
from pathlib import Path

def patient_name_key(value):
    # Metadata may append a group suffix, a space, or a parenthetical note.
    return re.split(r"[_\s（(]", str(value).strip(), maxsplit=1)[0]

# Only folders that actually contain a PNG count as available cases.
png_patient_folders = {group: set() for group in png_path_dict}
for group, paths in png_path_dict.items():
    for path in paths:
        parts = Path(os.path.relpath(path, root_directory)).parts
        if len(parts) >= 3 and parts[0] == group:
            png_patient_folders[group].add(parts[1])

def has_matched_png(group, patient_name):
    name = patient_name_key(patient_name)
    if not name:
        return False
    # The next character may be a folder annotation, but not another name character.
    pattern = re.compile(rf"^{re.escape(name)}(?=$|[\s（(0-9])")
    return any(pattern.search(folder) for folder in png_patient_folders.get(group, ()))

matched_png_mask = pd.Series(
    [has_matched_png(group, name) for group, name in
     zip(meta_data_excluded["Group"], meta_data_excluded["PatientName"])],
    index=meta_data_excluded.index,
)

case_count_before_png_check = len(meta_data_excluded)
excluded_png_cases = meta_data_excluded.loc[
    ~matched_png_mask, ["PatientName", "Group"]
].copy()
meta_data_excluded = meta_data_excluded.loc[matched_png_mask].copy()

assert len(meta_data_excluded) + len(excluded_png_cases) == case_count_before_png_check
print("Remaining cases by group:")
remaining_counts = meta_data_excluded["Group"].value_counts()
for group in png_path_dict:
    print(f"{group}: {remaining_counts.get(group, 0)}")

print(f"\nExcluded cases without matched PNG files ({len(excluded_png_cases)}):")
print(excluded_png_cases.to_string() if not excluded_png_cases.empty else "None")

Remaining cases by group:
TP: 250
TN: 250
FP: 200
FN: 200

Excluded cases without matched PNG files (21):
    PatientName Group
102         任伟娟    TP
271         朱仙娣    TN
322         蒋莹莹    TN
338         张雪艳    TN
350         余晓燕    TN
353          闵雪    TN
354          吴珍    TN
357         鲁礼媛    TN
367          吴珺    TN
372          丁可    TN
381          李洁    TN
400         郐世伟    TN
402         范文珺    TN
439         耿文静    TN
465         张小露    TN
515         谈吉敏    TN
730         潘小芹    FN
810         刘志敏    FN
820          沈娟    FN
824          于燕    FN
898         肖惠敏    FN


In [139]:
meta_data_excluded['GroundTruth'] = [1 if row['Group'] in ['TP', 'FN'] else 0 for idx, row in meta_data_excluded.iterrows()]

In [140]:
# meta_data_excluded.columns
print(meta_data_excluded['GroundTruth'].value_counts())
print(meta_data_excluded.columns.to_list())
meta_data_excluded.head()

GroundTruth
1    450
0    450
Name: count, dtype: int64
['PatientName', 'PatientAge', 'DensityCategory', 'LesionType', 'BIRADSRisk', 'HistologicalSubtype', 'Group', 'OriginalPatientName', 'DensityCategory_std', 'DensityCategory_num', 'BIRADSRisk_std', 'BIRADSRisk_sub', 'LesionType_cat', 'Histology_category', 'Histology_uncertain_reason', 'Density_std', 'Density_coarse', 'BIRADS_main', 'BIRADS_coarse', 'BIRADS_binary', 'LesionType_coarse', 'GroundTruth']


,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,OriginalPatientName,DensityCategory_std,DensityCategory_num,...,LesionType_cat,Histology_category,Histology_uncertain_reason,Density_std,Density_coarse,BIRADS_main,BIRADS_coarse,BIRADS_binary,LesionType_coarse,GroundTruth
0,崔敏,41,c,肿块伴簇状钙化,4B,浸润性导管癌,TP,崔敏（左）,C,3.0,...,Mixed,IDC / NST,,C,High density (C/D),4.0,4 Suspicious,Positive (≥4),Mixed lesions,1
1,顾璟怡,69,c,肿块伴簇状钙化,6,浸润性导管癌,TP,顾璟怡,C,3.0,...,Mixed,IDC / NST,,C,High density (C/D),6.0,6 Known malignancy,Positive (≥4),Mixed lesions,1
2,李彩琴,58,c,结构紊乱,4C,大汗腺型导管原位癌,TP,李彩琴,C,3.0,...,ArchitecturalDistortion/Asymmetry,DCIS,,C,High density (C/D),4.0,4 Suspicious,Positive (≥4),Architectural distortion/asymmetry,1
3,付启红,55,c,肿块,4B,混合性腺癌,TP,付启红,C,3.0,...,Mass,Other malignancies,,C,High density (C/D),4.0,4 Suspicious,Positive (≥4),Mass/Nodule dominant,1
4,周夏琴,49,b,细小钙化灶,4C,导管原位癌伴微小浸润,TP,周夏琴,B,2.0,...,Calcification,Other malignancies,,B,Low density (A/B),4.0,4 Suspicious,Positive (≥4),Calcification dominant,1


In [141]:
meta_data_sampled_TN = meta_data_excluded[meta_data_excluded['Group'] == 'TN'].copy()
meta_data_sampled_TP = meta_data_excluded[meta_data_excluded['Group'] == 'TP'].copy()
meta_data_sampled_FN = meta_data_excluded[meta_data_excluded['Group'] == 'FN'].copy()
meta_data_sampled_FP = meta_data_excluded[meta_data_excluded['Group']=='FP'].copy()
meta_data_sampled = pd.concat([meta_data_sampled_TN, meta_data_sampled_TP, 
                               meta_data_sampled_FN, meta_data_sampled_FP], ignore_index=True)
assert len(meta_data_sampled) == len(meta_data_excluded)


In [145]:
print(meta_data_sampled["Group"].value_counts())

malignant_histology = meta_data_sampled.loc[
    meta_data_sampled["Group"].isin(["TP", "FN"])
].copy()
assert len(malignant_histology) == 450
assert malignant_histology["GroundTruth"].eq(1).all()
assert meta_data_sampled.loc[
    ~meta_data_sampled["Group"].isin(["TP", "FN"]), "Histology_category"
].eq(HISTOLOGY_NOT_APPLICABLE).all()
assert malignant_histology["Histology_category"].isin(HISTOLOGY_CATEGORIES).all()

# Table 1
table1_histology_summary = (
    malignant_histology["Histology_category"]
    .value_counts()
    .reindex(HISTOLOGY_CATEGORIES, fill_value=0)
    .rename_axis("Histology_category")
    .reset_index(name="n")
)
table1_histology_summary["N"] = len(malignant_histology)
table1_histology_summary["percent"] = (
    100 * table1_histology_summary["n"] / table1_histology_summary["N"]
).round(1)
assert table1_histology_summary["n"].sum() == 450

print("Recorded histology in TP/FN cohort (N=450; mutually exclusive):")
display(table1_histology_summary)
print("Records with unavailable or uncertain histology wording:")
display(
    malignant_histology.loc[
        malignant_histology["Histology_category"].eq(
            "Histology unavailable / uncertain"
        ),
        ["PatientName", "Group", "HistologicalSubtype", "Histology_uncertain_reason"],
    ]
)


Group
TN    250
TP    250
FN    200
FP    200
Name: count, dtype: int64
Recorded histology in TP/FN cohort (N=450; mutually exclusive):


,Histology_category,n,N,percent
0,IDC / NST,337,450,74.9
1,DCIS,22,450,4.9
2,Other malignancies,84,450,18.7
3,Histology unavailable / uncertain,7,450,1.6


Records with unavailable or uncertain histology wording:


,PatientName,Group,HistologicalSubtype,Histology_uncertain_reason
318,邵莹,TP,纤维腺瘤及乳腺病,Benign wording conflicts with TP/FN cohort label
340,吴丽华,TP,无手术病理,No histological diagnosis recorded in this field
550,沙捷亚,FN,NaN,Histology text missing
556,谭晓红,FN,我院未手术,No histological diagnosis recorded in this field
599,董雯娟,FN,导管内乳头状瘤伴导管上皮增生活跃,Benign wording conflicts with TP/FN cohort label
667,吴建芬,FN,低分化腺癌组织，考虑乳腺来源,"Carcinoma is recorded, but breast origin is un..."
696,顾文娟,FN,导管内乳头状病变，实性乳头状癌不能除外,Malignancy is not confirmed in this field


In [146]:
df = meta_data_sampled.copy()

columns = ['PatientAge', 'DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat', 
           'BIRADSRisk_sub', 'Density_coarse', 'BIRADS_main', 'BIRADS_coarse', 'BIRADS_binary', 'LesionType_coarse']
categorical = ['DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat', 
               'BIRADSRisk_sub', 'Density_coarse', 'BIRADS_main', 'BIRADS_coarse', 'BIRADS_binary', 'LesionType_coarse']

# Reorder categories for each categorical variable by frequency
for col in ['LesionType_cat', 'LesionType_coarse']:
    ordered_categories = df[col].value_counts(dropna=False).index
    df[col] = pd.Categorical(df[col], categories=ordered_categories, ordered=True)

nonnormal = None
mytable = TableOne(df, columns=columns, categorical=categorical, nonnormal=nonnormal, pval=False)

In [147]:
mytable

Missing      Overall
n                                                                              900
PatientAge, mean (SD)                                               0  47.9 (12.4)
DensityCategory_std, n (%) A                                              31 (3.4)
                           B                                            134 (14.9)
                           C                                            640 (71.1)
                           D                                             95 (10.6)
BIRADSRisk_std, n (%)      0.0                                             2 (0.2)
                           1.0                                            36 (4.0)
                           2.0                                          128 (14.2)
                           3.0                                          286 (31.8)
                           4.0                                          420 (46.7)
                           5.0                                            23 (2.6)
                           6.0                                             5 (0.6)
LesionType_cat, n (%)      Mass                                         361 (40.1)
                           Calcification                                179 (19.9)
                           Mixed                                        160 (17.8)
                           Negative/Benign                                76 (8.4)
                           Nodule/FocalDensity                            52 (5.8)
                           ArchitecturalDistortion/Asymmetry              42 (4.7)
                           Unknown/NA                                     29 (3.2)
                           Other                                           1 (0.1)
BIRADSRisk_sub, n (%)      4A                                           223 (24.8)
                           4B                                           107 (11.9)
                           4C                                             87 (9.7)
                           None                                         483 (53.7)
Density_coarse, n (%)      High density (C/D)                           735 (81.7)
                           Low density (A/B)                            165 (18.3)
BIRADS_main, n (%)         0.0                                             2 (0.2)
                           1.0                                            36 (4.0)
                           2.0                                          128 (14.2)
                           3.0                                          286 (31.8)
                           4.0                                          420 (46.7)
                           5.0                                            23 (2.6)
                           6.0                                             5 (0.6)
BIRADS_coarse, n (%)       0 Incomplete                                    2 (0.2)
                           1–2 Negative/Benign                          164 (18.2)
                           3 Probably benign                            286 (31.8)
                           4 Suspicious                                 420 (46.7)
                           5 Highly suggestive                            23 (2.6)
                           6 Known malignancy                              5 (0.6)
BIRADS_binary, n (%)       Negative (<4)                                452 (50.2)
                           Positive (≥4)                                448 (49.8)
LesionType_coarse, n (%)   Mass/Nodule dominant                         413 (45.9)
                           Calcification dominant                       179 (19.9)
                           Mixed lesions                                160 (17.8)
                           Negative/Benign/None                           76 (8.4)
                           Architectural distortion/asymmetry             42 (4.7)
                           Other/Unknown                                  30 (

In [148]:
df = meta_data_sampled.copy()

columns = ['PatientAge', 'DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat']
categorical = ['DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat']

# Reorder categories for each categorical variable by frequency
for col in ['LesionType_cat']:
    ordered_categories = df[col].value_counts(dropna=False).index
    df[col] = pd.Categorical(df[col], categories=ordered_categories, ordered=True)

nonnormal = None
mytable = TableOne(df, columns=columns, categorical=categorical, 
                   nonnormal=nonnormal, pval=True, groupby='Group')

In [149]:
mytable

Grouped by Group                                                                         
                                                                      Missing      Overall           FN           FP           TN           TP P-Value
n                                                                                      900          200          200          250          250        
PatientAge, mean (SD)                                                       0  47.9 (12.4)  50.5 (11.3)  44.9 (11.7)  42.5 (10.7)  53.7 (12.3)  <0.001
DensityCategory_std, n (%) A                                                      31 (3.4)      7 (3.5)      2 (1.0)      5 (2.0)     17 (6.8)  <0.001
                           B                                                    134 (14.9)    33 (16.5)    34 (17.0)     23 (9.2)    44 (17.6)        
                           C                                                    640 (71.1)   147 (73.5)   151 (75.5)   166 (66.4)   176 (70.4)        
                           D                                                     95 (10.6)     13 (6.5)     13 (6.5)    56 (22.4)     13 (5.2)        
BIRADSRisk_std, n (%)      0.0                                                     2 (0.2)      0 (0.0)      0 (0.0)      0 (0.0)      2 (0.8)  <0.001
                           1.0                                                    36 (4.0)     10 (5.0)      0 (0.0)    26 (10.4)      0 (0.0)        
                           2.0                                                  128 (14.2)    71 (35.5)      0 (0.0)    57 (22.8)      0 (0.0)        
                           3.0                                                  286 (31.8)   119 (59.5)      0 (0.0)   167 (66.8)      0 (0.0)        
                           4.0                                                  420 (46.7)      0 (0.0)  200 (100.0)      0 (0.0)   220 (88.0)        
                           5.0                                                    23 (2.6)      0 (0.0)      0 (0.0)      0 (0.0)     23 (9.2)        
                           6.0                                                     5 (0.6)      0 (0.0)      0 (0.0)      0 (0.0)      5 (2.0)        
LesionType_cat, n (%)      Mass                                                 361 (40.1)     12 (6.0)   102 (51.0)   118 (47.2)   129 (51.6)  <0.001
                           Calcification                                        179 (19.9)    70 (35.0)    27 (13.5)    47 (18.8)    35 (14.0)        
                           Mixed                                                160 (17.8)    36 (18.0)    33 (16.5)     20 (8.0)    71 (28.4)        
                           Negative/Benign                                        76 (8.4)    56 (28.0)      0 (0.0)     20 (8.0)      0 (0.0)        
                           Nodule/FocalDensity                                    52 (5.8)    26 (13.0)    24 (12.0)      2 (0.8)      0 (0.0)        
                           ArchitecturalDistortion/Asymmetry                      42 (4.7)      0 (0.0)     14 (7.0)     22 (8.8)      6 (2.4)        
                           Unknown/NA                                             29 (3.2)      0 (0.0)      0 (0.0)     21 (8.4)      8 (3.2)        
                           Other                                                   1 (0.1)      0 (0.0)      0 (0.0)      0 (0.0)      1 (0.4)

## PNG paths


In [150]:
# 一张 PNG 一行：由最终病例表与现有 PNG 文件直接建立对应关系。
from pathlib import Path
import re

png_root = Path(root_directory).resolve()
final_metadata = meta_data_sampled.copy()
assert final_metadata["PatientName"].is_unique
assert final_metadata["OriginalPatientName"].notna().all()

# 使用原始姓名匹配患者目录；括号或空格后的内容是名单中的备注。
def original_name_key(value):
    return re.split(r"[\s（(]", str(value).strip(), maxsplit=1)[0]

png_records = []
for group, paths in png_path_dict.items():
    for path in paths:
        png_path = Path(path).resolve()
        parts = png_path.relative_to(png_root).parts
        if len(parts) >= 3 and parts[0] == group:
            png_records.append((group, parts[1], png_path))

index_rows = []
for row in final_metadata.itertuples(index=False):
    name = original_name_key(row.OriginalPatientName)
    folder_pattern = re.compile(rf"^{re.escape(name)}(?=$|[\s（(0-9])")
    for group, patient_folder, png_path in png_records:
        if group == row.Group and folder_pattern.search(patient_folder):
            index_rows.append({
                "PatientName": row.PatientName,
                "OriginalPatientName": row.OriginalPatientName,
                "Group": row.Group,
                "PNGPath": str(png_path),
            })

case_png_index = pd.DataFrame(
    index_rows,
    columns=["PatientName", "OriginalPatientName", "Group", "PNGPath"],
)
print(f"病例数: {len(final_metadata)}; PNG 对应记录数: {len(case_png_index)}")
display(case_png_index.head())


病例数: 900; PNG 对应记录数: 3851


,PatientName,OriginalPatientName,Group,PNGPath
0,徐琴芳,徐琴芳,TN,/Users/hannn/Documents/论文数据/data/preprocessed_...
1,徐琴芳,徐琴芳,TN,/Users/hannn/Documents/论文数据/data/preprocessed_...
2,徐琴芳,徐琴芳,TN,/Users/hannn/Documents/论文数据/data/preprocessed_...
3,徐琴芳,徐琴芳,TN,/Users/hannn/Documents/论文数据/data/preprocessed_...
4,沈诗怡,沈诗怡,TN,/Users/hannn/Documents/论文数据/data/preprocessed_...


In [151]:
# 检查每个入选病例均有 PNG；图片张数仅报告，不据此删除病例。
case_png_counts = (
    case_png_index.groupby("PatientName").size().rename("PNGCount")
)
case_png_summary = final_metadata[
    ["PatientName", "OriginalPatientName", "Group"]
].copy()
case_png_summary["PNGCount"] = (
    case_png_summary["PatientName"].map(case_png_counts).fillna(0).astype(int)
)
print("各组病例数:")
print(final_metadata["Group"].value_counts().reindex(["TP", "TN", "FP", "FN"], fill_value=0))
print(f"有 PNG 的病例: {(case_png_summary['PNGCount'] > 0).sum()}")
print(f"无 PNG 的病例: {(case_png_summary['PNGCount'] == 0).sum()}")
print("每例 PNG 张数分布:")
print(case_png_summary["PNGCount"].value_counts().sort_index())
print("PNG 张数不为 4 的病例:")
display(case_png_summary.loc[case_png_summary["PNGCount"] != 4])

assert case_png_summary["PNGCount"].gt(0).all(), "最终病例中存在没有 PNG 的病例"
assert set(case_png_index["PatientName"]) == set(final_metadata["PatientName"])
assert not final_metadata.loc[
    final_metadata["Group"].eq("FP"), "OriginalPatientName"
].isin(fp_exclusion_log["PatientName"]).any(), "核正后应剔除的 FP 病例仍在最终表中"

# 新文件使用 UTF-8；后续可直接用 pandas.read_csv 读取。
output_dir = png_root.parent / "meta_data"
output_dir.mkdir(parents=True, exist_ok=True)
final_metadata_path = output_dir / "meta_data_final.csv"
case_png_index_path = output_dir / "case_png_index.csv"
final_metadata.to_csv(final_metadata_path, index=False, encoding="utf-8")
case_png_index.to_csv(case_png_index_path, index=False, encoding="utf-8")
print(f"已保存: {final_metadata_path}")
print(f"已保存: {case_png_index_path}")


各组病例数:
Group
TP    250
TN    250
FP    200
FN    200
Name: count, dtype: int64
有 PNG 的病例: 900
无 PNG 的病例: 0
每例 PNG 张数分布:
PNGCount
1        2
2       17
3        4
4      852
5       18
6        1
8        2
11       1
12       1
101      1
159      1
Name: count, dtype: int64
PNG 张数不为 4 的病例:


,PatientName,OriginalPatientName,Group,PNGCount
56,顾玉荣,顾玉荣,TN,8
97,瞿晓凤,瞿晓凤,TN,3
148,张学如,张学如,TN,2
261,陆菊萍,陆菊萍,TP,2
285,褚春妹,褚春妹,TP,5
328,刘颖_TP1,刘颖,TP,2
344,安淑贤,安淑贤,TP,2
369,刘葛粉,刘葛粉,TP,3
408,王辉珍,王辉珍,TP,6
452,柳雪萍,柳雪萍,TP,2


已保存: /Users/hannn/Documents/论文数据/data/meta_data/meta_data_final.csv
已保存: /Users/hannn/Documents/论文数据/data/meta_data/case_png_index.csv
